# Maintenance Intelligence

This notebook integrates supervised priority prediction, complaint history, recurring issue signals, affected-user impact, and unsupervised clustering to generate an actionable maintenance ranking.


In [ ]:
import pandas as pd
import numpy as np
import joblib

df = pd.read_csv("../data/processed/campuspulse_akgec_cleaned.csv")

metadata = joblib.load("../ml/models/classification/model_metadata.pkl")
model = joblib.load("../ml/models/classification/final_priority_model.pkl")

print(df.shape)


In [ ]:
df["complaint_date"] = pd.to_datetime(df["complaint_date"], errors="coerce")
df["complaint_month"] = df["complaint_date"].dt.month
df["complaint_dayofweek"] = df["complaint_date"].dt.dayofweek
df["complaint_year"] = df["complaint_date"].dt.year
df["is_weekend"] = (df["complaint_dayofweek"] >= 5).astype(int)

df["complaint_time"] = pd.to_datetime(df["complaint_time"], errors="coerce")
df["complaint_hour"] = df["complaint_time"].dt.hour

df["time_period"] = pd.cut(
    df["complaint_hour"],
    bins=[-1, 5, 11, 17, 21, 24],
    labels=["Night", "Morning", "Afternoon", "Evening", "Late_Night"]
)

df["total_recent_complaints"] = (
    df["complaints_last_7_days"].fillna(0)
    + df["complaints_last_30_days"].fillna(0)
)

df["complaint_growth"] = (
    df["complaints_last_7_days"].fillna(0)
    / (df["complaints_last_30_days"].fillna(0) + 1)
)

df["impact_score"] = (
    df["affected_users"].fillna(0)
    * (1 + df["recurrence_count"].fillna(0))
)

df["duration_per_user"] = (
    df["issue_duration_hours"].fillna(0)
    / (df["affected_users"].fillna(0) + 1)
)

df["repair_cost_per_user"] = (
    df["estimated_repair_cost"].fillna(0)
    / (df["affected_users"].fillna(0) + 1)
)

df["equipment_infrastructure_age_gap"] = (
    df["infrastructure_age_years"].fillna(0)
    - df["equipment_age_years"].fillna(0)
)

df["operational_pressure"] = (
    df["complaints_last_7_days"].fillna(0)
    + df["recurrence_count"].fillna(0)
    + df["affected_users"].fillna(0) / 10
)

print("Feature engineering completed.")
print(df.shape)


In [ ]:
preprocessor = joblib.load(
    "../ml/models/classification/campuspulse_preprocessor.pkl"
)

pca = joblib.load(
    "../ml/models/classification/campuspulse_pca.pkl"
)

model = joblib.load(
    "../ml/models/classification/final_priority_model.pkl"
)

feature_columns = list(preprocessor.feature_names_in_)

X = df[feature_columns].copy()

print("X shape:", X.shape)
print("Features:", len(X.columns))


In [ ]:
X_processed = preprocessor.transform(X)

X_pca = pca.transform(X_processed)

predicted_priority = model.predict(X_pca)

df["predicted_priority"] = predicted_priority

print(df["predicted_priority"].value_counts())


## Complaint Intelligence Signals

The maintenance ranking uses predicted priority along with complaint similarity, recurrence history, and affected-user impact.


In [ ]:
priority_map = {
    "Low": 1,
    "Medium": 2,
    "High": 3,
    "Critical": 4
}

df["priority_score"] = df["predicted_priority"].map(priority_map)

df["similar_complaint_count"] = (
    df["previous_similar_complaints"].fillna(0)
    + df["complaints_last_7_days"].fillna(0)
    + df["complaints_last_30_days"].fillna(0)
)

df["recurring_issue_signal"] = (
    df["recurrence_count"].fillna(0)
    + df["previous_similar_complaints"].fillna(0)
)

df["affected_user_impact"] = df["affected_users"].fillna(0)

print(df[[
    "predicted_priority",
    "similar_complaint_count",
    "recurring_issue_signal",
    "affected_user_impact"
]].head())


In [ ]:
def normalize(series):
    min_val = series.min()
    max_val = series.max()

    if max_val == min_val:
        return pd.Series(0, index=series.index)

    return (series - min_val) / (max_val - min_val)

df["priority_norm"] = normalize(df["priority_score"])
df["similar_norm"] = normalize(df["similar_complaint_count"])
df["recurring_norm"] = normalize(df["recurring_issue_signal"])
df["affected_norm"] = normalize(df["affected_user_impact"])

print("Signals normalized successfully.")


## K-Means Complaint Pattern Integration

K-Means clustering is used to incorporate recurring complaint patterns identified during the unsupervised learning stage.

The clustering features represent complaint impact, duration, recurrence, complaint frequency, equipment age, infrastructure age, and estimated repair cost.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pca_features = [
    "affected_users",
    "issue_duration_hours",
    "previous_similar_complaints",
    "complaints_last_7_days",
    "complaints_last_30_days",
    "recurrence_count",
    "equipment_age_years",
    "infrastructure_age_years",
    "estimated_repair_cost"
]

X_cluster = df[pca_features].copy()
X_cluster = X_cluster.fillna(X_cluster.median())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

cluster_pca = PCA(n_components=6)
pca_data_transformed = cluster_pca.fit_transform(X_scaled)

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(pca_data_transformed)

print("Cluster distribution:")
print(df["cluster"].value_counts().sort_index())


In [ ]:
cluster_counts = df["cluster"].value_counts()

df["cluster_similar_count"] = df["cluster"].map(cluster_counts)
df["cluster_similar_norm"] = normalize(df["cluster_similar_count"])

df["similar_pattern_norm"] = (
    0.5 * df["similar_norm"]
    + 0.5 * df["cluster_similar_norm"]
)

print(df[[
    "cluster",
    "similar_complaint_count",
    "cluster_similar_count",
    "similar_pattern_norm"
]].head())


## Final Maintenance Score

The final maintenance score combines:

- Predicted Priority ? 40%
- Similar/Cluster Complaint Pattern ? 20%
- Recurring Issue Signal ? 20%
- Affected User Impact ? 20%

All signals are normalized before combining them. The resulting score is a decision-ranking score, not an ML probability.


In [ ]:
df["maintenance_score"] = (
    0.40 * df["priority_norm"]
    + 0.20 * df["similar_pattern_norm"]
    + 0.20 * df["recurring_norm"]
    + 0.20 * df["affected_norm"]
)

df["maintenance_rank"] = (
    df["maintenance_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

print("Maintenance score calculated successfully.")


In [ ]:
top_10 = df.sort_values(
    "maintenance_score",
    ascending=False
).head(10)

print(top_10[[
    "complaint_id",
    "predicted_priority",
    "cluster",
    "similar_complaint_count",
    "cluster_similar_count",
    "recurring_issue_signal",
    "affected_user_impact",
    "maintenance_score",
    "maintenance_rank"
]].to_string(index=False))


## Maintenance Intelligence Summary

The maintenance intelligence layer combines supervised ML priority prediction with complaint history, recurring-issue signals, affected-user impact, and K-Means clustering.

The final maintenance score provides a single ranking mechanism for identifying complaints that require faster maintenance attention.

Higher maintenance scores indicate higher maintenance priority based on the combined signals.

The approach integrates both supervised and unsupervised learning outputs into an actionable complaint-ranking system.
